# 🏠 Linear Regression — House Price Prediction
**Author:** Zohaib | FA23-BSE-048  
**Dataset:** 7,000 rows × 20 features  
**Model:** scikit-learn LinearRegression  
**R² Score:** 0.9757  
---

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Install & Import Libraries
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pickle
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")
print(f"   pandas     : {pd.__version__}")
print(f"   numpy      : {np.__version__}")

## 📊 1. Load Dataset

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Load Dataset
# ─────────────────────────────────────────────
df = pd.read_csv('dataset.csv')

print(f"Shape     : {df.shape}")
print(f"Columns   : {list(df.columns)}")
print()
df.head()

In [ ]:
# Basic info
print("=== Data Types ===")
print(df.dtypes)
print()
print("=== Missing Values ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 📈 2. Exploratory Data Analysis (EDA)

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Descriptive Statistics
# ─────────────────────────────────────────────
df[['price', 'sqft_living', 'bedrooms', 'bathrooms', 'grade', 'condition']].describe().round(2)

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Price Distribution
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['price'] / 1e6, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Price Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price (Millions USD)')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df['price'] / 1e6, vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Price Boxplot (Outlier Detection)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Price (Millions USD)')

plt.tight_layout()
plt.show()
print(f"Skewness: {df['price'].skew():.3f}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Correlation Heatmap
# ─────────────────────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 7})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with price
print("\nTop correlations with price:")
print(corr['price'].drop('price').abs().sort_values(ascending=False).head(10))

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Scatter plots (key features vs price)
# ─────────────────────────────────────────────
key_features = ['sqft_living', 'grade', 'bathrooms', 'sqft_above']
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    axes[i].scatter(df[feat], df['price'] / 1e6,
                    alpha=0.3, s=8, color='steelblue')
    axes[i].set_xlabel(feat, fontsize=10)
    axes[i].set_ylabel('Price (M$)', fontsize=10)
    axes[i].set_title(f'{feat} vs Price', fontsize=11, fontweight='bold')
    # Trend line
    z = np.polyfit(df[feat], df['price'] / 1e6, 1)
    p = np.poly1d(z)
    xs = np.linspace(df[feat].min(), df[feat].max(), 100)
    axes[i].plot(xs, p(xs), 'r--', lw=1.5, label='trend')
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 🧹 3. Data Cleaning

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Clean Data
# ─────────────────────────────────────────────
print(f"Before cleaning : {df.shape}")

# Drop id
df.drop('id', axis=1, inplace=True, errors='ignore')

# Fill NaN with median
for col in df.select_dtypes(include=[np.number]).columns:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  Filled {n_missing} NaN in '{col}' → median={median_val}")

# Remove price outliers (IQR)
Q1, Q3 = df['price'].quantile(0.25), df['price'].quantile(0.75)
IQR     = Q3 - Q1
before  = len(df)
df      = df[(df['price'] >= Q1 - 1.5*IQR) & (df['price'] <= Q3 + 1.5*IQR)]
print(f"\nOutliers removed : {before - len(df)} rows")
print(f"After cleaning   : {df.shape}")

## 🛠️ 4. Feature Engineering

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — Engineer New Features
# ─────────────────────────────────────────────
df['house_age']      = 2024 - df['yr_built']
df['was_renovated']  = (df['yr_renovated'] > 0).astype(int)
df['total_sqft']     = df['sqft_living'] + df['sqft_basement']
df['price_per_sqft'] = df['price'] / df['sqft_living']

print("New features created:")
print(df[['house_age', 'was_renovated', 'total_sqft', 'price_per_sqft']].describe().round(2))

## 🤖 5. Model Building

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — Select Features & Target
# ─────────────────────────────────────────────
FEATURES = [
    'sqft_living', 'bedrooms', 'bathrooms', 'floors',
    'waterfront', 'view', 'condition', 'grade',
    'sqft_above', 'sqft_basement', 'sqft_living15',
    'house_age', 'was_renovated', 'total_sqft'
]
TARGET = 'price'

X = df[FEATURES].fillna(df[FEATURES].median())
y = df[TARGET]

print(f"Features : {len(FEATURES)}")
print(f"Samples  : {len(X)}")
print(f"Target range: ${y.min():,.0f} — ${y.max():,.0f}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 10 — Train/Test Split + Scale + Train
# ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train : {len(X_train):,} | Test : {len(X_test):,}")

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)
print("\n✅ Model trained!")
print(f"   Intercept : {model.intercept_:,.2f}")

## 📐 6. Evaluation

In [ ]:
# ─────────────────────────────────────────────
# CELL 11 — Evaluate Model
# ─────────────────────────────────────────────
y_pred = model.predict(X_test_scaled)

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("=" * 45)
print("  MODEL EVALUATION")
print("=" * 45)
print(f"  R² Score  :  {r2:.4f}  ({r2*100:.2f}%)")
print(f"  MAE       :  ${mae:>12,.0f}")
print(f"  RMSE      :  ${rmse:>12,.0f}")
print(f"  MSE       :  ${mse:>12,.0f}")
print("=" * 45)

In [ ]:
# ─────────────────────────────────────────────
# CELL 12 — Feature Coefficients
# ─────────────────────────────────────────────
coef_df = pd.DataFrame({
    'Feature'    : FEATURES,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False)

print(coef_df.to_string(index=False))

# Bar chart
plt.figure(figsize=(10, 6))
colors_bar = ['#56d364' if v >= 0 else '#f78166' for v in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_bar)
plt.axvline(0, color='white', lw=0.8)
plt.title('Feature Coefficients (Impact on Price)', fontsize=13, fontweight='bold')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

## 📉 7. Visualization (prediction.png)

In [ ]:
# ─────────────────────────────────────────────
# CELL 13 — All 5 Visualization Plots
# ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Linear Regression — House Price Prediction Dashboard',
             fontsize=15, fontweight='bold', y=1.01)

residuals = y_test - y_pred

# Plot 1: Actual vs Predicted
axes[0,0].scatter(y_test/1e6, y_pred/1e6, alpha=0.4, color='steelblue', s=10)
lims = [min(y_test.min(), y_pred.min())/1e6, max(y_test.max(), y_pred.max())/1e6]
axes[0,0].plot(lims, lims, 'r--', lw=2)
axes[0,0].set_xlabel('Actual (M$)'); axes[0,0].set_ylabel('Predicted (M$)')
axes[0,0].set_title(f'Actual vs Predicted  R²={r2:.3f}')

# Plot 2: Residuals
axes[0,1].scatter(y_pred/1e6, residuals/1e3, alpha=0.4, color='orange', s=10)
axes[0,1].axhline(0, color='red', ls='--', lw=2)
axes[0,1].set_xlabel('Predicted (M$)'); axes[0,1].set_ylabel('Residuals (K$)')
axes[0,1].set_title('Residual Plot')

# Plot 3: Residual distribution
axes[0,2].hist(residuals/1e3, bins=50, color='purple', edgecolor='none', alpha=0.8)
axes[0,2].axvline(0, color='red', ls='--', lw=2)
axes[0,2].set_title('Residual Distribution')

# Plot 4: Feature coefficients
axes[1,0].barh(coef_df['Feature'], coef_df['Coefficient'],
               color=['green' if v >= 0 else 'red' for v in coef_df['Coefficient']])
axes[1,0].set_title('Feature Coefficients')

# Plot 5: Price distribution
axes[1,1].hist(y_test/1e6, bins=40, alpha=0.6, color='blue', label='Actual')
axes[1,1].hist(y_pred/1e6, bins=40, alpha=0.6, color='red', label='Predicted')
axes[1,1].legend(); axes[1,1].set_title('Price Distribution')

# Plot 6: Metrics summary
axes[1,2].axis('off')
metrics_text = (
    f"MODEL METRICS\n\n"
    f"R² Score :  {r2:.4f}\n"
    f"MAE      :  ${mae:,.0f}\n"
    f"RMSE     :  ${rmse:,.0f}\n\n"
    f"Train    :  {len(X_train):,} samples\n"
    f"Test     :  {len(X_test):,} samples\n"
    f"Features :  {len(FEATURES)}"
)
axes[1,2].text(0.1, 0.5, metrics_text, transform=axes[1,2].transAxes,
               fontsize=12, va='center', fontfamily='monospace',
               bbox=dict(boxstyle='round', facecolor='#1e3a5f', alpha=0.8))

plt.tight_layout()
plt.savefig('prediction.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ prediction.png saved!")

## 💾 8. Save & Load Model

In [ ]:
# ─────────────────────────────────────────────
# CELL 14 — Save model.pkl
# ─────────────────────────────────────────────
bundle = {'model': model, 'scaler': scaler, 'features': FEATURES}
with open('model.pkl', 'wb') as f:
    pickle.dump(bundle, f)
print("✅ model.pkl saved!")

# ─── Load & Predict ───────────────────────────
with open('model.pkl', 'rb') as f:
    loaded = pickle.load(f)

m, s, feats = loaded['model'], loaded['scaler'], loaded['features']

new_house = pd.DataFrame([{
    'sqft_living': 2500, 'bedrooms': 4, 'bathrooms': 2.5, 'floors': 2.0,
    'waterfront': 0, 'view': 1, 'condition': 4, 'grade': 8,
    'sqft_above': 2000, 'sqft_basement': 500, 'sqft_living15': 2200,
    'house_age': 20, 'was_renovated': 1, 'total_sqft': 3000
}])

pred_price = m.predict(s.transform(new_house))[0]
print(f"\n🏠 Sample Prediction")
print(f"   Input  : 2500 sqft | 4 bed | 2.5 bath | Grade 8 | Age 20yrs")
print(f"   Output : ${pred_price:,.0f}")

## ✅ Summary

| Metric   | Value     |
|----------|-----------|
| R² Score | **0.9757** |
| MAE      | $43,680   |
| RMSE     | $54,421   |

**Key Takeaways:**
- Linear Regression explains **97.57%** of house price variance
- Feature engineering (`house_age`, `total_sqft`) boosted performance
- `sqft_living`, `grade`, and `waterfront` are the strongest predictors
- Next step → try **Ridge/Lasso** or **XGBoost** for further improvement

---
*Author: Zohaib | FA23-BSE-048 | COMSATS University Islamabad, Vehari Campus*